<div style="font-family: Calibri; background-color: #ccd5ae; padding: 10px 10px 10px 10px;">
    <h1>Feature Selection</h1>
    <h3>- Filters</h3>
</div>

<p><strong><span style="font-size: 20px;">Variance Threshold</span></strong></p>
<ul>
    <li>Removes features with low variance (features that don&apos;t change much).</li>
    <li>Use <span style="color: rgb(41, 105, 176);">sklearn.feature_selection.VarianceThreshold</span>.</li>
    <li>When to Use: To remove features that are nearly constant and provide little information.</li>
</ul>

In [115]:
import pandas as pd
import numpy as np

# Loading dataset 
[Dataset source - fetch_california_housing](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html#sklearn.datasets.fetch_california_housing)


In [116]:
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()

In [117]:
X =  pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

In [118]:
X.sample()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
11779,5.8718,6.0,6.505963,0.936968,1738.0,2.960818,38.77,-121.28


In [140]:
X.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


# Check the existing variance in data

In [121]:
_variance = X.var()
df_coef = pd.DataFrame({"coef":np.round(_variance, 2)})

In [122]:
df_coef.sort_values("coef")

,coef
AveBedrms,0.22
MedInc,3.61
Longitude,4.01
Latitude,4.56
AveRooms,6.12
AveOccup,107.87
HouseAge,158.40
Population,1282470.46


# Check the existing variance of data
Without any other preprocessing

In [103]:
from sklearn.feature_selection import VarianceThreshold

In [123]:
selector = VarianceThreshold(threshold=5.0)  # Remove features with variance < 5.0

In [124]:
selected_features = selector.fit_transform(X)

In [130]:
(df_coef.loc[X.columns[selector.get_support()]]).sort_values(by='coef')

,coef
AveRooms,6.12
AveOccup,107.87
HouseAge,158.40
Population,1282470.46


# Normalize Data 

### When Should You Normalize Before VarianceThreshold?
<ol>
    <li>If Features Are on Very Different Scales<ul>
            <li>Example: If one feature has values between 0-1 and another between 1000-5000, their variances will be very different.</li>
            <li>Applying VarianceThreshold on raw data might remove only low-scale features, which may not be ideal.</li>
            <li>Normalization ensures fair variance comparisons.</li>
        </ul>
    </li>
    <li>If You Plan to Use Other Feature Selection Methods Afterward<ul>
            <li>Some methods (e.g., PCA, L1-based selection) work better when features are normalized.</li>
            <li>Normalization before VarianceThreshold can ensure smoother integration with other techniques.</li>
        </ul>
    </li>
</ol>

In [132]:
from sklearn.preprocessing import MinMaxScaler

In [133]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [134]:
_variance = pd.DataFrame(X_scaled, columns=X.columns).var()
df_coef = pd.DataFrame({"coef":np.round(_variance, 2)})

In [135]:
df_coef

,coef
MedInc,0.02
HouseAge,0.06
AveRooms,0.00
AveBedrms,0.00
Population,0.00
AveOccup,0.00
Latitude,0.05
Longitude,0.04


In [136]:
selector = VarianceThreshold(threshold=0.01)

In [137]:
X_selected_scaled = selector.fit_transform(X_scaled)
df_coef.loc[X.columns[selector.get_support()]]

,coef
MedInc,0.02
HouseAge,0.06
Latitude,0.05
Longitude,0.04


### Observations
When the data was not normalized